# Deep-Guard — Large-Scale AI Media Detection Training

Trains one model to answer a single question about **any** uploaded
image or video frame: *was this made by AI, or captured by a camera?*
Not faces-only. Not one generator. Any subject.

## What this trains on

Two datasets are combined, giving roughly **200,000 images** spanning
five different generator families:

| Source | Images | What it contributes |
|---|---|---|
| `tristanzhang32/ai-generated-images-vs-real-images` | 60,000 | AI from **Stable Diffusion, MidJourney, DALL-E**. Real from Pexels, Unsplash, WikiArt. General subjects: scenery, animals, objects, artwork |
| `xhlulu/140k-real-and-fake-faces` | 140,000 | AI faces from **StyleGAN** (a GAN, not a diffusion model). Real faces from Flickr/FFHQ |

**Why combine rather than pick the biggest one?** Published research on
this exact problem is consistent: detectors trained on a single
generator learn *that generator's* quirks instead of a general sense of
"AI-ness," and collapse to near-random accuracy on generators they have
never seen. Generator *diversity* matters more than raw image count.
Combining diffusion models with a GAN, and faces with general scenes, is
what gives the model a chance to generalise to generators that did not
exist when it was trained.

## Why this also covers AI-generated video

Modern AI video tools (Sora, Runway, Kling, Veo) are diffusion models —
the same underlying technology as Stable Diffusion and MidJourney, just
extended over time. Individual frames pulled from their output carry the
same statistical fingerprints as diffusion-generated still images. So a
strong general image detector transfers to AI-generated video frames,
which is exactly how your app already analyses video: sample frames,
score each one, aggregate.

**Honest limitation to state in your report:** face-*swap* deepfakes
(a real video with someone else's face pasted on) are a different
problem. They leave blending seams at the face boundary rather than
whole-image generation artifacts. This model will be weaker on those.
Covering them properly needs a face-swap video dataset such as
FaceForensics++, which is a sensible next milestone.

## Also fixed here

The original notebook's fine-tuning stage only unfroze ~10% of the
network, at a learning rate so low it barely moved — which is why
accuracy was stuck at 85.67%. This notebook fine-tunes the whole
network properly.

---

### Before you run

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Expect **2-4 hours** total. Colab disconnects after ~90 minutes of
   *browser* inactivity, so leave the tab open and check in periodically.
3. Step 3 offers to mount your Google Drive. **Say yes.** Checkpoints get
   copied there after every improvement, so a disconnect costs you time,
   not work.

## Step 1 — Check the GPU and available disk

In [ ]:
!nvidia-smi
print()
!df -h /content | tail -1
print()
print("You need roughly 60 GB free under /content for both datasets.")
print("If free space is under 60 GB, set USE_LARGE_DATASET_ONLY = True in Step 4.")

## Step 2 — Install the Kaggle tool

In [ ]:
!pip install -q kaggle==1.6.17

## Step 3 — Credentials and Drive backup

Two things here: your Kaggle key (so the datasets can download), and
optionally your Google Drive (so long training runs survive a
disconnect). Mounting Drive opens a permission popup — approve it.

In [ ]:
from google.colab import files

print("Select your kaggle.json file below.")
uploaded = files.upload()

assert "kaggle.json" in uploaded, (
    "No file named kaggle.json was uploaded. Re-run this cell and pick the "
    "correct file. It must be named exactly kaggle.json."
)

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle credentials installed.")

In [ ]:
# Strongly recommended for a multi-hour run.
USE_DRIVE_BACKUP = True

DRIVE_BACKUP_PATH = None
if USE_DRIVE_BACKUP:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_BACKUP_PATH = "/content/drive/MyDrive/deepguard_bouncer.pth"
        print(f"\nCheckpoints will be backed up to: {DRIVE_BACKUP_PATH}")
    except Exception as e:
        print(f"Drive mount failed ({e}). Continuing without backup.")
        DRIVE_BACKUP_PATH = None
else:
    print("Drive backup disabled. A disconnect will lose your progress.")

## Step 4 — Configuration

Two switches you may want to change:

- `USE_LARGE_DATASET_ONLY` — set to `True` if Step 1 showed less than
  60 GB free. Trains on the 140k faces set only (smaller download), at
  the cost of general-subject coverage.
- `MAX_PER_CLASS` — caps how many images are used per class. `None` uses
  everything (best accuracy, longest run). Set it to e.g. `40000` to cut
  training time roughly in half.

In [ ]:
USE_LARGE_DATASET_ONLY = False   # True = skip the 52 GB general dataset
MAX_PER_CLASS = None             # e.g. 40000 to cap, None = use everything

EPOCHS_HEAD = 2                  # warm-up epochs (frozen backbone)
EPOCHS_FINETUNE = 6              # full fine-tuning epochs
BATCH_SIZE = 96

print(f"General dataset:  {'SKIPPED' if USE_LARGE_DATASET_ONLY else 'INCLUDED'}")
print(f"Images per class: {'all available' if MAX_PER_CLASS is None else MAX_PER_CLASS}")
print(f"Epochs:           {EPOCHS_HEAD} warm-up + up to {EPOCHS_FINETUNE} fine-tune")

## Step 5 — Download the datasets

Each zip is deleted immediately after extraction, so peak disk usage
stays as low as possible. The large one is ~52 GB and can take 15-30
minutes; the faces one is ~4 GB.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

def free_gb():
    return shutil.disk_usage("/content").free / 1e9

def fetch(slug, folder_name):
    target = DATA_DIR / folder_name
    if target.exists() and any(target.iterdir()):
        print(f"[skip] {slug} already present at {target}")
        return
    target.mkdir(parents=True, exist_ok=True)
    print(f"\n[download] {slug}  (free disk: {free_gb():.1f} GB)")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", slug, "-p", str(target)],
        check=True,
    )
    print(f"[extract]  {slug}")
    for zip_path in target.glob("*.zip"):
        subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(target)], check=True)
        zip_path.unlink()          # free the space immediately
    print(f"[done]     {slug}  (free disk: {free_gb():.1f} GB)")

if not USE_LARGE_DATASET_ONLY:
    fetch("tristanzhang32/ai-generated-images-vs-real-images", "general")

fetch("xhlulu/140k-real-and-fake-faces", "faces")

print(f"\nAll downloads complete. Free disk remaining: {free_gb():.1f} GB")

## Step 6 — Find the class folders automatically

Different datasets name their folders differently (`REAL`/`FAKE`,
`real`/`fake`, `ai_images`, and so on). Rather than hardcode a guess that
silently breaks everything, this scans what was actually extracted and
reports what it found.

**Read this output before continuing.** Both classes must appear.

In [ ]:
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

REAL_WORDS = ["real", "authentic", "human", "nature"]
FAKE_WORDS = ["fake", "ai", "synthetic", "generated", "artificial"]

def classify_folder(name):
    low = name.lower()
    for w in FAKE_WORDS:
        if low == w or low.startswith(w + "_") or low.endswith("_" + w) or w in low.split("_"):
            return "fake"
    for w in REAL_WORDS:
        if low == w or low.startswith(w + "_") or low.endswith("_" + w) or w in low.split("_"):
            return "real"
    return None

def count_images_direct(folder):
    """Counts images directly inside `folder` (not recursively), so nested
    class folders aren't double-counted by their parents."""
    try:
        return sum(1 for f in os.scandir(folder)
                   if f.is_file() and Path(f.name).suffix.lower() in IMAGE_EXT)
    except OSError:
        return 0

discovered = []
for root, dirs, _ in os.walk(DATA_DIR):
    label = classify_folder(Path(root).name)
    if label is None:
        continue
    n = count_images_direct(root)
    if n > 0:
        discovered.append((root, label, n))

print("Class folders found:\n")
total_real = total_fake = 0
for path, label, n in sorted(discovered):
    print(f"  [{label:4s}] {os.path.relpath(path, DATA_DIR):55s} {n:>7,} images")
    if label == "real":
        total_real += n
    else:
        total_fake += n

print(f"\n  TOTAL real: {total_real:,}")
print(f"  TOTAL fake: {total_fake:,}")
print(f"  TOTAL:      {total_real + total_fake:,}")

assert total_real > 0 and total_fake > 0, (
    "One class has zero images — folder detection failed. Run the next cell "
    "and share its output."
)

In [ ]:
# Diagnostic only — run this if the cell above found nothing or looked wrong.
for root, dirs, fnames in os.walk(DATA_DIR):
    depth = len(Path(root).relative_to(DATA_DIR).parts)
    if depth > 3:
        dirs[:] = []
        continue
    n = sum(1 for f in fnames if Path(f).suffix.lower() in IMAGE_EXT)
    print("  " * depth + f"{Path(root).name}/  ({n} images directly here)")

## Step 7 — Setup and the label-order safeguard

`ImageFolder` assigns class numbers by sorting folder names
alphabetically, which would put `fake` before `real` and silently invert
every prediction the model ever makes. The dataset class below builds its
file list from explicit folder paths instead, so the mapping is fixed by
us, not by alphabetical accident.

In [ ]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True   # scraped datasets contain partial files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CLASS_ORDER = ["real", "fake"]   # index 0 = real, index 1 = fake
INPUT_SIZE = 224
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]


class FileListDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            # A few unreadable files is normal at this scale. Substitute
            # neutral grey rather than crashing a 3-hour training run.
            image = Image.new("RGB", (256, 256), (128, 128, 128))
        if self.transform:
            image = self.transform(image)
        return image, label


def build_model(pretrained: bool = False) -> nn.Module:
    """Must stay identical to app/model.py on your laptop."""
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)
    return model


train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

print("Setup complete.")

## Step 8 — Build a balanced train / validation / test split

Two things happen here that matter:

**Balancing.** The combined pool won't have equal real and fake counts.
An unbalanced set lets the model score well by favouring whichever class
is larger, so we trim both classes to the same size.

**Splitting by file, with a fixed random seed.** The test set is chosen
once and never seen during training, so the final accuracy is honest.

In [ ]:
random.seed(42)

pool = {"real": [], "fake": []}
for folder, label, _ in discovered:
    for entry in os.scandir(folder):
        if entry.is_file() and Path(entry.name).suffix.lower() in IMAGE_EXT:
            pool[label].append(entry.path)

for label in pool:
    random.shuffle(pool[label])

# Balance the two classes, and apply the optional cap.
limit = min(len(pool["real"]), len(pool["fake"]))
if MAX_PER_CLASS is not None:
    limit = min(limit, MAX_PER_CLASS)

samples = []
for label in CLASS_ORDER:
    for path in pool[label][:limit]:
        samples.append((path, CLASS_ORDER.index(label)))

random.shuffle(samples)

n_total = len(samples)
n_train = int(n_total * 0.80)
n_valid = int(n_total * 0.10)

train_samples = samples[:n_train]
valid_samples = samples[n_train:n_train + n_valid]
test_samples  = samples[n_train + n_valid:]

train_dataset = FileListDataset(train_samples, train_transforms)
valid_dataset = FileListDataset(valid_samples, eval_transforms)
test_dataset  = FileListDataset(test_samples,  eval_transforms)

print(f"Available per class after balancing: {limit:,}")
print(f"Total images used:                   {n_total:,}")
print(f"  Train: {len(train_dataset):,}")
print(f"  Valid: {len(valid_dataset):,}")
print(f"  Test:  {len(test_dataset):,}")

def label_balance(sample_list):
    counts = {0: 0, 1: 0}
    for _, lab in sample_list:
        counts[lab] += 1
    return f"real={counts[0]:,} fake={counts[1]:,}"

print(f"\nTrain balance: {label_balance(train_samples)}")
print(f"Test balance:  {label_balance(test_samples)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

## Step 9 — Build the model

The printout below is worth screenshotting for your report: it shows the
original bug in plain numbers.

In [ ]:
model = build_model(pretrained=True).to(device)

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def freeze_backbone(m):
    for p in m.features.parameters():
        p.requires_grad = False

def unfreeze_everything(m):
    for p in m.parameters():
        p.requires_grad = True

total = sum(p.numel() for p in model.parameters())
print(f"Total parameters:              {total:,}")

freeze_backbone(model)
head_only = count_trainable(model)
print(f"Phase A trainable (head only): {head_only:,}  ({head_only/total*100:.2f}%)")

for p in model.features[-1].parameters():
    p.requires_grad = True
old_phase_b = count_trainable(model)
print(f"OLD notebook Phase B:          {old_phase_b:,}  ({old_phase_b/total*100:.2f}%)  <-- the bug")

unfreeze_everything(model)
print(f"THIS notebook Phase B:         {count_trainable(model):,}  (100.00%)  <-- the fix")

freeze_backbone(model)
criterion = nn.BCEWithLogitsLoss()

## Step 10 — Training loop

Uses mixed precision, which roughly halves the time per epoch on a T4 with no loss of accuracy. Progress prints every 100 batches so you can tell it's alive during long epochs.

In [ ]:
import time
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")

def run_epoch(model, loader, optimizer=None, scheduler=None, tag=""):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    started = time.time()
    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for i, (images, labels) in enumerate(loader):
            images = images.to(device, non_blocking=True)
            labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with autocast("cuda"):
                logits = model(images)
                loss = criterion(logits, labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * images.size(0)
            predictions = (torch.sigmoid(logits.float()) >= 0.5).float()
            correct += (predictions == labels).sum().item()
            total += images.size(0)

            if is_train and i > 0 and i % 100 == 0:
                elapsed = time.time() - started
                pct = 100 * i / len(loader)
                eta = elapsed / i * (len(loader) - i)
                print(f"    {tag} batch {i}/{len(loader)} ({pct:.0f}%) "
                      f"running_acc={correct/total:.4f} eta={eta/60:.1f}min")

    return total_loss / total, correct / total


CHECKPOINT_PATH = "/content/deepguard_bouncer.pth"

def save_checkpoint(model):
    torch.save(model.state_dict(), CHECKPOINT_PATH)
    if DRIVE_BACKUP_PATH:
        try:
            shutil.copy(CHECKPOINT_PATH, DRIVE_BACKUP_PATH)
        except Exception as e:
            print(f"    (Drive backup failed: {e})")

print("Training functions ready.")

## Step 11 — Phase A: warm up the classifier head

The new head starts random. Training it briefly against a frozen backbone stops the first fine-tuning gradients from being large and destructive.

In [ ]:
freeze_backbone(model)

head_optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
best_val_acc = 0.0

for epoch in range(1, EPOCHS_HEAD + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, head_optimizer, tag=f"A{epoch}")
    val_loss, val_acc = run_epoch(model, valid_loader)
    print(f"[Phase A][{epoch}/{EPOCHS_HEAD}] train_acc={train_acc:.4f} | "
          f"val_acc={val_acc:.4f} | {(time.time()-t0)/60:.1f} min")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(model)
        print(f"  -> best so far ({val_acc:.4f}), saved.")

## Step 12 — Phase B: fine-tune the whole network

This is the cell that produces the accuracy. Every layer unfreezes, the
learning rate is 1e-4 (ten times the original notebook's), and a cosine
schedule decays it smoothly toward zero so the model settles instead of
oscillating.

**This is the long part.** Each epoch takes roughly 15-35 minutes
depending on the dataset size you chose. Progress prints every 100
batches with an ETA.

In [ ]:
unfreeze_everything(model)

finetune_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    finetune_optimizer, T_max=EPOCHS_FINETUNE * len(train_loader)
)

patience, stalled = 3, 0

for epoch in range(1, EPOCHS_FINETUNE + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(model, train_loader, finetune_optimizer,
                                      scheduler, tag=f"B{epoch}")
    val_loss, val_acc = run_epoch(model, valid_loader)
    lr_now = finetune_optimizer.param_groups[0]["lr"]
    print(f"[Phase B][{epoch}/{EPOCHS_FINETUNE}] train_acc={train_acc:.4f} | "
          f"val_acc={val_acc:.4f} | lr={lr_now:.2e} | {(time.time()-t0)/60:.1f} min")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        stalled = 0
        save_checkpoint(model)
        print(f"  -> best so far ({val_acc:.4f}), saved.")
    else:
        stalled += 1
        print(f"  -> no improvement ({stalled}/{patience})")
        if stalled >= patience:
            print("  -> stopping early.")
            break

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

## Step 13 — Final test on data never seen during training

This is the number you quote to your teacher.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))

test_loss, test_acc = run_epoch(model, test_loader)
print(f"TEST ACCURACY: {test_acc:.4f}   ({test_acc*100:.2f}%)")
print(f"Test set size: {len(test_dataset):,} images never seen during training")
print(f"\nPrevious model: 85.67%, and it only worked on human faces.")

## Step 14 — Detailed breakdown for your report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

model.eval()
all_probs, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        with autocast("cuda"):
            logits = model(images)
        all_probs.extend(torch.sigmoid(logits.float()).cpu().numpy().flatten().tolist())
        all_labels.extend(labels.numpy().tolist())

all_preds = [1 if p >= 0.5 else 0 for p in all_probs]
cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix (rows = actual, cols = predicted), order [real, fake]:")
print(cm)
print(f"\nReal images correctly identified: {tn:,}")
print(f"Real images wrongly called AI:    {fp:,}   (false alarms)")
print(f"AI images wrongly called real:    {fn:,}   (missed fakes)")
print(f"AI images correctly identified:   {tp:,}")
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_ORDER, digits=4))
print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}")

## Step 15 — Save and download

In [ ]:
import datetime

sources = ["xhlulu/140k-real-and-fake-faces"]
if not USE_LARGE_DATASET_ONLY:
    sources.insert(0, "tristanzhang32/ai-generated-images-vs-real-images")

torch.save({
    "model_state_dict": model.state_dict(),
    "class_order": CLASS_ORDER,
    "architecture": "efficientnet_b0",
    "input_size": INPUT_SIZE,
    "normalize_mean": NORMALIZE_MEAN,
    "normalize_std": NORMALIZE_STD,
    "trained_on": sources,
    "scope": "general AI-generated media detection (any subject, not faces-only)",
    "generators_covered": ["Stable Diffusion", "MidJourney", "DALL-E", "StyleGAN"],
    "training_images": n_total,
    "test_accuracy": test_acc,
    "training_version": 4,
    "saved_at_utc": datetime.datetime.utcnow().isoformat(),
}, CHECKPOINT_PATH)

if DRIVE_BACKUP_PATH:
    shutil.copy(CHECKPOINT_PATH, DRIVE_BACKUP_PATH)
    print(f"Backed up to Drive: {DRIVE_BACKUP_PATH}")

print(f"Saved. Test accuracy: {test_acc:.4f}  |  Trained on {n_total:,} images")

In [ ]:
from google.colab import files
files.download(CHECKPOINT_PATH)
print("Replace models/deepguard_bouncer.pth on your laptop with this file.")